In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
import kagglehub
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

from sklearn.model_selection import GridSearchCV , RandomizedSearchCV
from sklearn.pipeline import Pipeline

# These two are separate libraries, not part of sklearn - need pip install if not already available
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.metrics import precision_recall_curve

In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load



# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md


# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv


In [3]:
df = pd.read_csv("/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors='coerce')
df["TotalCharges"] = df["TotalCharges"].fillna(0)
pd.set_option("display.max_columns",None)

# print (df.head(10))
# print (df.info())

In [4]:
def feature_engineering(x_train , x_test):
    encoder = OneHotEncoder(sparse_output = False , handle_unknown ="ignore").set_output(transform="pandas")
    x_train_encoded = encoder.fit_transform(x_train)
    x_test_encoded = encoder.transform(x_test)
    return x_train_encoded , x_test_encoded

In [5]:
def feature_scaling(x_train,x_test):
    std = StandardScaler()
    x_train_scaled = pd.DataFrame(std.fit_transform(x_train),columns=x_train.columns,index=x_train.index)
    x_test_scaled = pd.DataFrame(std.transform(x_test),columns=x_test.columns,index=x_test.index)

    return x_train_scaled,x_test_scaled


In [6]:
yes_no_cols = [col for col in df.columns if set(df[col].dropna().unique()) == {'Yes', 'No'}]
print(yes_no_cols)

for col in yes_no_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']


In [7]:
def predict_at_threshold(model, x, threshold, is_svm=False):
    if is_svm:
        scores = model.decision_function(x)
    else:
        scores = model.predict_proba(x)[:, 1]
    return (scores >= threshold).astype(int)

In [8]:
def find_best_threshold_for_model(model, x_test, y_test):
    y_proba = model.predict_proba(x_test)[:, 1]
    precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba)
    
    # F1 for every threshold (drop the last precision/recall pair — 
    # sklearn adds an extra point with no corresponding threshold)
    f1_scores = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-10)
    
    best_idx = np.argmax(f1_scores)
    best_threshold = thresholds[best_idx]
    best_f1 = f1_scores[best_idx]
    
    return best_threshold, best_f1, precisions[best_idx], recalls[best_idx]

In [9]:
def find_best_threshold_svm(model, x_val, y_val):
    scores = model.decision_function(x_val)
    precisions, recalls, thresholds = precision_recall_curve(y_val, scores)
    f1_scores = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-10)
    best_idx = np.argmax(f1_scores)
    return thresholds[best_idx], f1_scores[best_idx], precisions[best_idx], recalls[best_idx]

In [10]:

def Classification(models_dict,x_train,y_train,x_test,y_test):
    
    # Split training data further into train + validation
    x_tr, x_val, y_tr, y_val = train_test_split(
    x_train, y_train, test_size=0.2, random_state=42, stratify=y_train
    )

    best_thresholds = {}
    for name, model in models_dict.items():
        model.fit(x_tr, y_tr)   # refit on the smaller train split
        if name == "SVM":
            threshold, f1, precision, recall = find_best_threshold_svm(model, x_val, y_val)
        else:
            threshold, f1, precision, recall = find_best_threshold_for_model(model, x_val, y_val)
        best_thresholds[name] = threshold
        print(f"{name}: best threshold = {threshold:.3f} (found on validation set)")
    
    # Now refit each model on FULL training data, and evaluate on test set using discovered thresholds
    for name, model in models_dict.items():
        
        model.fit(x_train, y_train)   # refit on full training data
        
        threshold = best_thresholds[name]

        is_svm = (name == "SVM")
        y_pred_test = predict_at_threshold(model, x_test, threshold=threshold, is_svm=is_svm)
        
        # For train accuracy, also apply the SAME threshold for a fair comparison
        y_pred_train = predict_at_threshold(model, x_train, threshold=threshold, is_svm=is_svm)
        train_acc = accuracy_score(y_train, y_pred_train)
        
        test_acc = accuracy_score(y_test, y_pred_test)
        precision = precision_score(y_test, y_pred_test)
        recall = recall_score(y_test, y_pred_test)
        f1 = f1_score(y_test, y_pred_test)
        
        print(f"--- {name} (threshold={threshold:.3f}) ---")
        print(f"Train Accuracy : {train_acc:.4f}")
        print(f"Test Accuracy  : {test_acc:.4f}")
        print(f"Precision      : {precision:.4f}")
        print(f"Recall         : {recall:.4f}")
        print(f"F1 Score       : {f1:.4f}")
        print(f"Confusion Matrix:\n{confusion_matrix(y_test, y_pred_test)}\n")
        


In [11]:
   
x_train,x_test, y_train , y_test = train_test_split (df.drop(columns=["Churn","customerID"]),df["Churn"],random_state=42)

    
x_train_numeric = x_train.select_dtypes(include=['int64','float64'])
x_train_variable = x_train.select_dtypes(include=['object'])

x_test_numeric = x_test.select_dtypes(include=['int64','float64'])
x_test_variable = x_test.select_dtypes(include=['object'])

x_train_variable_encoded,x_test_variable_encoded = feature_engineering(x_train_variable,x_test_variable)

x_train_encoded = pd.concat([x_train_variable_encoded,x_train_numeric],axis=1)
x_test_encoded = pd.concat([x_test_variable_encoded,x_test_numeric],axis=1)

print(x_train_encoded.shape)
# print(df.corr(numeric_only=True)['Churn'].sort_values(ascending=False))

x_train_scaled,x_test_scaled = feature_scaling(x_train_encoded,x_test_encoded)

# Used to check columns with null values
# print(x_train_scaled.isnull().sum())


(5282, 41)


In [12]:
# pipeline = Pipeline([
#     ('model', RandomForestClassifier(random_state=42, class_weight='balanced'))
# ])

# param_grid = {
#         'model__max_depth': [ 3, 4, 5, 6, 7, 8, 9, 10 , None],
#         'model__min_samples_split': [2, 3, 4, 5,6,7,8,9,10],
#         'model__min_samples_leaf': [1, 2, 4, 8],
#         'model__max_features': ['sqrt', 'log2', None],
#         'model__n_estimators': [100, 200]
#             }

# #Find Out parameters for max_depth and min_samples_split for Random Forest Regressor
# grid_search = GridSearchCV(
#     estimator=pipeline,
#     param_grid=param_grid,
#     cv=5,
#     scoring='f1',
#     n_jobs=-1,
#     )

# grid_search.fit(x_train_scaled, y_train)
# RDF_best_model = grid_search.best_estimator_.named_steps['model']
# print("Best parameters found: ", grid_search.best_params_)




In [13]:
# model = {
#     "Logistic Regression":LogisticRegression(random_state=42, class_weight='balanced', max_iter=1000),
#     "KNN": KNeighborsClassifier(),
#     "Naive Bayes": GaussianNB(),
#     "Decision Tree": DecisionTreeClassifier(random_state=42),
#     "Random Forest": RandomForestClassifier(**RDF_best_model.get_params()),
#     "Gradient Boosting": GradientBoostingClassifier(random_state=42),
#     "SVM": SVC(random_state=42),
#     "XGBoost": XGBClassifier(random_state=42, eval_metric='logloss'),
#     "LightGBM": LGBMClassifier(random_state=42),
#     "Neural Network (MLP)": MLPClassifier(random_state=42, max_iter=400)
# }
# Classification(model,x_train_scaled,y_train,x_test_scaled,y_test)

In [14]:


SCORING = 'recall'   # use 'f1' if truly binary
CV = 5
N_JOBS = -1

best_models = {}

# ---------------------------------------------------------
# 1. Logistic Regression
# ---------------------------------------------------------
pipe = Pipeline([('model', LogisticRegression(random_state=42, class_weight='balanced', max_iter=1000))])
param_grid = {
    'model__C': [0.01, 0.1, 1, 10, 100],
    'model__penalty': ['l2'],
    'model__solver': ['lbfgs', 'liblinear']
}
gs = GridSearchCV(pipe, param_grid, cv=CV, scoring=SCORING, n_jobs=N_JOBS)
gs.fit(x_train_scaled, y_train)
best_models["Logistic Regression"] = gs.best_estimator_.named_steps['model']
print("Logistic Regression best params:", gs.best_params_)

# ---------------------------------------------------------
# 2. KNN
# ---------------------------------------------------------
pipe = Pipeline([('model', KNeighborsClassifier())])
param_grid = {
    'model__n_neighbors': [3, 5, 7, 9, 11, 15],
    'model__weights': ['uniform', 'distance'],
    'model__p': [1, 2]   # 1=manhattan, 2=euclidean
}
gs = GridSearchCV(pipe, param_grid, cv=CV, scoring=SCORING, n_jobs=N_JOBS)
gs.fit(x_train_scaled, y_train)
best_models["KNN"] = gs.best_estimator_.named_steps['model']
print("KNN best params:", gs.best_params_)

# ---------------------------------------------------------
# 3. Naive Bayes (very few params to tune)
# ---------------------------------------------------------
pipe = Pipeline([('model', GaussianNB())])
param_grid = {
    'model__var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6, 1e-5]
}
gs = GridSearchCV(pipe, param_grid, cv=CV, scoring=SCORING, n_jobs=N_JOBS)
gs.fit(x_train_scaled, y_train)
best_models["Naive Bayes"] = gs.best_estimator_.named_steps['model']
print("Naive Bayes best params:", gs.best_params_)

# ---------------------------------------------------------
# 4. Decision Tree
# ---------------------------------------------------------
pipe = Pipeline([('model', DecisionTreeClassifier(random_state=42, class_weight='balanced'))])
param_grid = {
    'model__max_depth': [3, 5, 7, 9, 11, None],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4],
    'model__criterion': ['gini', 'entropy']
}
gs = GridSearchCV(pipe, param_grid, cv=CV, scoring=SCORING, n_jobs=N_JOBS)
gs.fit(x_train_scaled, y_train)
best_models["Decision Tree"] = gs.best_estimator_.named_steps['model']
print("Decision Tree best params:", gs.best_params_)

# ---------------------------------------------------------
# 5. Random Forest (your original block)
# ---------------------------------------------------------
pipe = Pipeline([('model', RandomForestClassifier(random_state=42, class_weight='balanced'))])
param_grid = {
    'model__max_depth': [3, 4, 5, 6, 7, 8, 9, 10, None],
    'model__min_samples_split': [2, 3, 4, 5, 6, 7, 8, 9, 10],
    'model__min_samples_leaf': [1, 2, 4, 8],
    'model__max_features': ['sqrt', 'log2', None],
    'model__n_estimators': [100, 200]
}
gs = RandomizedSearchCV(pipe, param_grid, n_iter=40, cv=CV, scoring=SCORING,
                         n_jobs=N_JOBS, random_state=42)  # RandomizedSearchCV: grid is huge (1944 combos)
gs.fit(x_train_scaled, y_train)
best_models["Random Forest"] = gs.best_estimator_.named_steps['model']
print("Random Forest best params:", gs.best_params_)

# ---------------------------------------------------------
# 6. Gradient Boosting
# ---------------------------------------------------------
pipe = Pipeline([('model', GradientBoostingClassifier(random_state=42))])
param_grid = {
    'model__n_estimators': [100, 200, 300],
    'model__learning_rate': [0.01, 0.05, 0.1, 0.2],
    'model__max_depth': [3, 4, 5, 6],
    'model__subsample': [0.8, 1.0]
}
gs = RandomizedSearchCV(pipe, param_grid, n_iter=30, cv=CV, scoring=SCORING,
                         n_jobs=N_JOBS, random_state=42)
gs.fit(x_train_scaled, y_train)
best_models["Gradient Boosting"] = gs.best_estimator_.named_steps['model']
print("Gradient Boosting best params:", gs.best_params_)

# ---------------------------------------------------------
# 7. SVM
# ---------------------------------------------------------
pipe = Pipeline([('model', SVC(random_state=42, class_weight='balanced' , probability=True))])
param_grid = {
    'model__C': [0.1, 1, 10, 100],
    'model__kernel': ['rbf', 'linear'],
    'model__gamma': ['scale', 'auto']
}
gs = RandomizedSearchCV(pipe, param_grid, n_iter=15, cv=CV, scoring=SCORING,
                         n_jobs=N_JOBS, random_state=42)  # SVC is slow, keep search small
gs.fit(x_train_scaled, y_train)
best_models["SVM"] = gs.best_estimator_.named_steps['model']
print("SVM best params:", gs.best_params_)

# ---------------------------------------------------------
# 8. XGBoost
# ---------------------------------------------------------
pipe = Pipeline([('model', XGBClassifier(random_state=42, eval_metric='logloss'))])
param_grid = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [3, 4, 5, 6, 8],
    'model__learning_rate': [0.01, 0.05, 0.1, 0.2],
    'model__subsample': [0.8, 1.0],
    'model__colsample_bytree': [0.8, 1.0]
}
gs = RandomizedSearchCV(pipe, param_grid, n_iter=40, cv=CV, scoring=SCORING,
                         n_jobs=N_JOBS, random_state=42)
gs.fit(x_train_scaled, y_train)
best_models["XGBoost"] = gs.best_estimator_.named_steps['model']
print("XGBoost best params:", gs.best_params_)

# ---------------------------------------------------------
# 9. LightGBM
# ---------------------------------------------------------
pipe = Pipeline([('model', LGBMClassifier(random_state=42))])
param_grid = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [-1, 5, 10, 15],
    'model__learning_rate': [0.01, 0.05, 0.1, 0.2],
    'model__num_leaves': [15, 31, 63]
}
gs = RandomizedSearchCV(pipe, param_grid, n_iter=40, cv=CV, scoring=SCORING,
                         n_jobs=N_JOBS, random_state=42)
gs.fit(x_train_scaled, y_train)
best_models["LightGBM"] = gs.best_estimator_.named_steps['model']
print("LightGBM best params:", gs.best_params_)

# ---------------------------------------------------------
# 10. Neural Network (MLP)
# ---------------------------------------------------------
pipe = Pipeline([('model', MLPClassifier(random_state=42, max_iter=1000 , early_stopping=True))])
param_grid = {
    'model__hidden_layer_sizes': [(50,), (100,), (50, 50), (100, 50)],
    'model__activation': ['relu', 'tanh'],
    'model__alpha': [0.0001, 0.001, 0.01],
    'model__learning_rate_init': [0.001, 0.01]
}
gs = RandomizedSearchCV(pipe, param_grid, n_iter=20, cv=CV, scoring=SCORING,
                         n_jobs=N_JOBS, random_state=42)  # MLP is slow, keep search small
gs.fit(x_train_scaled, y_train)
best_models["Neural Network (MLP)"] = gs.best_estimator_.named_steps['model']
print("MLP best params:", gs.best_params_)

# ---------------------------------------------------------
# Final: run your Classification comparison on all tuned models
# ---------------------------------------------------------
Classification(best_models, x_train_scaled, y_train, x_test_scaled, y_test)

Logistic Regression best params: {'model__C': 0.01, 'model__penalty': 'l2', 'model__solver': 'liblinear'}
KNN best params: {'model__n_neighbors': 15, 'model__p': 1, 'model__weights': 'uniform'}
Naive Bayes best params: {'model__var_smoothing': 1e-09}
Decision Tree best params: {'model__criterion': 'gini', 'model__max_depth': 3, 'model__min_samples_leaf': 1, 'model__min_samples_split': 2}
Random Forest best params: {'model__n_estimators': 200, 'model__min_samples_split': 6, 'model__min_samples_leaf': 8, 'model__max_features': 'log2', 'model__max_depth': 3}
Gradient Boosting best params: {'model__subsample': 0.8, 'model__n_estimators': 200, 'model__max_depth': 4, 'model__learning_rate': 0.2}
SVM best params: {'model__kernel': 'linear', 'model__gamma': 'scale', 'model__C': 0.1}
XGBoost best params: {'model__subsample': 1.0, 'model__n_estimators': 200, 'model__max_depth': 3, 'model__learning_rate': 0.05, 'model__colsample_bytree': 1.0}
[LightGBM] [Warning] Found whitespace in feature_names